# 01 Baseline and Neighbor Analysis

연구 목적별 통합 Notebook — 승인된 매핑, 원본 스크립트 및 결과는 변경하지 않는다. 1차 Fine-Tuning까지 포함. 2차 Fine-Tuning 제외.

## 연구 구성
- [`evaluate_handmade_baseline.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/KNN/Normal/scripts/evaluate_handmade_baseline.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/KNN/Normal/logs)
- [`analyze_knn_neighbors.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/KNN/Normal/scripts/analyze_knn_neighbors.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/KNN/Normal/logs)

## 사용법
원본 코드 셀은 `%%archive_source`로 등록하는 실제 원본 코드이며, 함수명 충돌을 막기 위해 현재 커널에서 임의 실행하지 않는다. 맨 아래의 실행 셀에서 원본 스크립트/Notebook을 고정 커밋의 **별도 checkout**에서 실행한다. 각 실험의 실행 인자·입력·사전 조건은 위 원본 로그/코드로 확인한다. 기존 로그·결과와 새 출력이 섞이지 않도록 독립 복사본을 사용한다. Original Test/교수님 Test를 포함한 파일은 실행 환경에 두지 않는다. 과거 스크립트가 해당 자료를 요구할 경우 실행하지 않는다. 자동 재실행 및 성능 동등성은 미검증.

원본 실행을 위한 로그 디렉터리를 별도로 지정한다.

In [ ]:
# 공통 실행 제어 — Run All은 원본 코드 등록만 하며 학습하지 않는다.
from pathlib import Path
import hashlib, json, subprocess, sys, datetime
from IPython import get_ipython
SOURCE_CELLS = {}
def archive_source(line, cell):
    SOURCE_CELLS[line.strip()] = cell
get_ipython().register_magic_function(archive_source, magic_kind="cell", magic_name="archive_source")
# 실행 시에만 설정: 기존 작업 트리가 아닌 임시 checkout (git worktree add --detach ... <ref>)
WORKTREE = None
LOG_DIR = None
RUN_PLAN = []  # 예: [("원본/스크립트.py", ["--인자", "값"])]
ALLOW_EXECUTION = False

def blob_sha(data):
    return hashlib.sha1(f"blob {len(data)}\0".encode()+data).hexdigest()

def get_verified_source(path):
    if WORKTREE is None:
        raise ValueError("해당 커밋의 별도 checkout 경로 WORKTREE를 지정하세요.")
    root=Path(WORKTREE).expanduser().resolve()
    entry=SOURCE_INDEX[path]
    head=subprocess.check_output(["git","-C",str(root),"rev-parse","HEAD"],text=True).strip()
    if head != entry["ref"]:
        raise ValueError(f"checkout ref 불일치: {head} != {entry['ref']}")
    p=root/path
    if blob_sha(p.read_bytes()) != entry["sha"]:
        raise ValueError(f"원본 Git blob 불일치: {path}")
    if path in SOURCE_CELLS:
        if p.read_text(encoding="utf-8-sig").rstrip("\n") != SOURCE_CELLS[path].rstrip("\n").lstrip("\ufeff"):
            raise ValueError(f"Notebook 보존 코드와 원본이 다릅니다: {path}")
    return root,p

def show_original(path):
    from IPython.display import display, Code
    _,p=get_verified_source(path)
    display(Code(p.read_text(encoding="utf-8"),language="python" if p.suffix==".py" else "json"))

def execute_original(path, args):
    if not ALLOW_EXECUTION:
        raise RuntimeError("명시적 실행 승인이 필요합니다.")
    root,p=get_verified_source(path)
    if LOG_DIR is None:
        raise ValueError("새 외부 LOG_DIR를 지정하세요.")
    logs=Path(LOG_DIR).expanduser().resolve();logs.mkdir(parents=True,exist_ok=True)
    if logs==root or root in logs.parents:
        raise ValueError("LOG_DIR는 checkout 밖에 두세요.")
    stamp=datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    if p.suffix==".ipynb":
        cmd=[sys.executable,"-m","jupyter","nbconvert","--to","notebook","--execute",str(p),"--output-dir",str(logs),"--output",f"{p.stem}_{stamp}.ipynb",*args]
    else:
        cmd=[sys.executable,str(p),*args]
    log=logs/f"{p.stem}_{stamp}.log"
    with log.open("x",encoding="utf-8") as f:
        f.write(json.dumps({"source":path,**SOURCE_INDEX[path],"command":cmd,"cwd":str(p.parent)},ensure_ascii=False)+"\n");f.flush()
        result=subprocess.run(cmd,cwd=p.parent,stdout=f,stderr=subprocess.STDOUT)
        f.write(f"\nreturncode={result.returncode}\n")
    print(log)
    result.check_returncode()


In [ ]:
SOURCE_INDEX = {'05_model_optimize/KNN/Normal/scripts/evaluate_handmade_baseline.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': '1101e16c097124b4f460f382ad6a8befb344173a'}, '05_model_optimize/KNN/Normal/scripts/analyze_knn_neighbors.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': '3b3859f84a8138efe168fc66e70e76417fc8de55'}}

## evaluate_handmade_baseline

원본: [05_model_optimize/KNN/Normal/scripts/evaluate_handmade_baseline.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/KNN/Normal/scripts/evaluate_handmade_baseline.py)  
Git blob: `1101e16c097124b4f460f382ad6a8befb344173a`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/KNN/Normal/logs)

KNN Normal baseline: evaluate the four standardized development handwriting sets.

Run from repository root:
python 05_model_optimize/KNN/Normal/scripts/evaluate_handmade_baseline.py

The official Original Test is never loaded. Hand-made is evaluation-only.



In [ ]:
%%archive_source 05_model_optimize/KNN/Normal/scripts/evaluate_handmade_baseline.py
#!/usr/bin/env python3
"""KNN Normal baseline: evaluate the four standardized development handwriting sets.

Run from repository root:
python 05_model_optimize/KNN/Normal/scripts/evaluate_handmade_baseline.py

The official Original Test is never loaded. Hand-made is evaluation-only.
"""
from __future__ import annotations

import csv
import hashlib
import json
import platform
import sys
import time
from pathlib import Path

import numpy as np
import sklearn
from sklearn.metrics import accuracy_score, f1_score

ROOT = Path(__file__).resolve().parents[4]
sys.path.insert(0, str(ROOT / "03_experiments" / "model_optimization"))
from common import fixed_split, pipeline, scores, LABELS
from knn import build_baseline

HANDMADE = ROOT / "02_data" / "Hand_made"
OUTPUT = ROOT / "05_model_optimize" / "KNN" / "Normal" / "results" / "EXP002_handmade_baseline"

EXPECTED_SHA256 = {
    41: "0a829172a72787c770139c73de34cc8b751181829ff6fe08ec260247b9be5a2f",
    42: "f364fb453d05b4fe2a705c2b8a80683b404608c7c257d664e3ec7ac86932d4c5",
    43: "10db20997c727c1bc1f1299f8e89626fadd3ec9618468d6759c86b0d2ddc4a66",
    44: "0e266a36e7b29edfcca0514836b5c383d13939948b4e335d70f61bb40bdc084d",
}


def write_csv(path, rows):
    with path.open("w", encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


def main():
    x_train, y_train, _, _, dataset_sha = fixed_split()
    estimator, use_scaling = build_baseline()
    model = pipeline(estimator, use_scaling)

    print("KNN Normal baseline training:", len(y_train), flush=True)
    start = time.perf_counter()
    model.fit(x_train, y_train)
    fit_seconds = time.perf_counter() - start

    y_all, pred_all = [], []
    writer_rows, prediction_rows = [], []
    predict_seconds = 0.0

    for writer_id in (41, 42, 43, 44):
        filepath = HANDMADE / f"ml26_{writer_id}.npz"
        digest = hashlib.sha256(filepath.read_bytes()).hexdigest()
        if digest != EXPECTED_SHA256[writer_id]:
            raise ValueError(f"Hand-made SHA256 mismatch for {writer_id}: {digest}")

        with np.load(filepath, allow_pickle=False) as data:
            if set(data.files) != {"img", "label", "label_id"}:
                raise ValueError(f"Unexpected keys: {filepath}: {data.files}")
            images = data["img"]
            label_ids = data["label_id"].astype(np.int64)
            labels = data["label"]

        if images.shape != (640, 28, 28) or images.dtype != np.uint8:
            raise ValueError(f"Unexpected image shape/dtype: {filepath}")
        if label_ids.shape != (640,):
            raise ValueError(f"Unexpected label shape: {filepath}")
        if not np.array_equal(
            label_ids, np.asarray([LABELS.index(str(c)) for c in labels], dtype=np.int64)
        ):
            raise ValueError(f"Incorrect label mapping: {filepath}")
        counts = np.bincount(label_ids, minlength=16)
        if not np.all(counts == 40):
            raise ValueError(f"Unexpected per-class counts: {filepath}")

        x = images.reshape(640, 784).astype(np.float32)
        start = time.perf_counter()
        predictions = model.predict(x).astype(np.int64)
        predict_seconds += time.perf_counter() - start

        correct = int(np.sum(predictions == label_ids))
        acc = accuracy_score(label_ids, predictions)
        macro = f1_score(label_ids, predictions, labels=np.arange(16),
                         average="macro", zero_division=0)
        writer_rows.append({
            "writer": writer_id, "n": 640, "correct": correct,
            "accuracy": float(acc), "macro_f1": float(macro)
        })
        print(f"Writer {writer_id}: {correct}/640, accuracy={acc:.6f}, macro_f1={macro:.6f}", flush=True)

        for i, (true, pred) in enumerate(zip(label_ids, predictions)):
            prediction_rows.append({
                "writer": writer_id, "index": i,
                "true": LABELS[int(true)], "predicted": LABELS[int(pred)],
                "correct": int(true == pred)
            })
        y_all.append(label_ids)
        pred_all.append(predictions)

    y_true = np.concatenate(y_all)
    y_pred = np.concatenate(pred_all)
    result = scores(y_true, y_pred)
    result.update({
        "model": "KNN Normal Baseline", "handmade_n": len(y_true),
        "official_train_n": len(y_train), "official_dataset_sha256": dataset_sha,
        "handmade_sha256": {str(k): v for k, v in EXPECTED_SHA256.items()},
        "fit_seconds": fit_seconds, "handmade_predict_seconds": predict_seconds,
        "python": platform.python_version(),
        "sklearn": sklearn.__version__, "numpy": np.__version__,
        "hyperparameters": {str(k): repr(v) for k, v in estimator.get_params().items()},
        "scaling": "pixel/255 in pipeline",
        "evaluation_role": "repeated-use development Hand-made; not independent final test",
        "writers": writer_rows
    })

    OUTPUT.mkdir(parents=True, exist_ok=True)
    (OUTPUT / "metrics.json").write_text(
        json.dumps(result, indent=2, ensure_ascii=False), encoding="utf-8"
    )
    write_csv(OUTPUT / "by_writer.csv", writer_rows)
    write_csv(OUTPUT / "predictions.csv", prediction_rows)
    cm = np.array(result["confusion_matrix"])
    write_csv(OUTPUT / "confusion_matrix.csv", [
        {"true/pred": LABELS[i], **{LABELS[j]: int(cm[i, j]) for j in range(16)}}
        for i in range(16)
    ])
    errors = [
        {"true": LABELS[i], "predicted": LABELS[j], "count": int(cm[i, j])}
        for i in range(16) for j in range(16)
        if i != j and cm[i, j]
    ]
    errors.sort(key=lambda row: (-row["count"], row["true"], row["predicted"]))
    write_csv(OUTPUT / "top_confusions.csv", errors)
    print(f"\nTotal: accuracy={result['accuracy']:.6f}, macro_f1={result['macro_f1']:.6f}")
    print("Top 10 confusions:")
    for row in errors[:10]:
        print(f"  {row['true']} -> {row['predicted']}: {row['count']}")
    print("Saved:", OUTPUT)


if __name__ == "__main__":
    main()


## analyze_knn_neighbors

원본: [05_model_optimize/KNN/Normal/scripts/analyze_knn_neighbors.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/KNN/Normal/scripts/analyze_knn_neighbors.py)  
Git blob: `3b3859f84a8138efe168fc66e70e76417fc8de55`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/KNN/Normal/logs)

EXP-003: Visualize Hand-made KNN misclassifications and their 5 nearest training neighbors.

Run from repository root:
  python 05_model_optimize/KNN/Normal/scripts/analyze_knn_neighbors.py

This script does NOT overwrite EXP-001/EXP-002, tune hyperparameters, or load official Test.



In [ ]:
%%archive_source 05_model_optimize/KNN/Normal/scripts/analyze_knn_neighbors.py
#!/usr/bin/env python3
"""EXP-003: Visualize Hand-made KNN misclassifications and their 5 nearest training neighbors.

Run from repository root:
  python 05_model_optimize/KNN/Normal/scripts/analyze_knn_neighbors.py

This script does NOT overwrite EXP-001/EXP-002, tune hyperparameters, or load official Test.
"""
from __future__ import annotations

import csv
import hashlib
import json
import sys
from collections import Counter
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

ROOT = Path(__file__).resolve().parents[4]
sys.path.insert(0, str(ROOT / "03_experiments" / "model_optimization"))
from common import LABELS, fixed_split, pipeline
from knn import build_baseline

HANDMADE = ROOT / "02_data" / "Hand_made"
BASELINE = ROOT / "05_model_optimize" / "KNN" / "Normal" / "results" / "EXP002_handmade_baseline"
OUTPUT = ROOT / "05_model_optimize" / "KNN" / "Normal" / "results" / "EXP003_error_neighbors"

EXPECTED_SHA256 = {
    41: "0a829172a72787c770139c73de34cc8b751181829ff6fe08ec260247b9be5a2f",
    42: "f364fb453d05b4fe2a705c2b8a80683b404608c7c257d664e3ec7ac86932d4c5",
    43: "10db20997c727c1bc1f1299f8e89626fadd3ec9618468d6759c86b0d2ddc4a66",
    44: "0e266a36e7b29edfcca0514836b5c383d13939948b4e335d70f61bb40bdc084d",
}
TARGET_PAIRS = [("0", "D"), ("9", "4"), ("7", "1"), ("8", "B")]
CASES_PER_PAIR = 4


def load_handmade():
    images, labels = {}, {}
    for writer in (41, 42, 43, 44):
        path = HANDMADE / f"ml26_{writer}.npz"
        actual_sha = hashlib.sha256(path.read_bytes()).hexdigest()
        if actual_sha != EXPECTED_SHA256[writer]:
            raise ValueError(f"SHA-256 mismatch for {path}: {actual_sha}")
        with np.load(path, allow_pickle=False) as obj:
            data = obj["img"]
            y = obj["label_id"]
            strings = obj["label"]
        if data.shape != (640, 28, 28) or data.dtype != np.uint8:
            raise ValueError(f"Unexpected images: {path} {data.shape} {data.dtype}")
        if len(y) != 640 or np.bincount(y.astype(int), minlength=16).tolist() != [40] * 16:
            raise ValueError(f"Unexpected label distribution: {path}")
        if not np.array_equal(y.astype(int), np.array([LABELS.index(str(s)) for s in strings])):
            raise ValueError(f"Label mismatch: {path}")
        images[writer], labels[writer] = data, y.astype(int)
    return images, labels


def select_cases(records, pair):
    selected = []
    pool = [r for r in records if (r["true"], r["predicted"]) == pair]
    for writer in (41, 42, 43, 44):
        options = [r for r in pool if r["writer"] == writer]
        if options:
            selected.append(options[0])
    for item in pool:
        if len(selected) == CASES_PER_PAIR:
            break
        if item not in selected:
            selected.append(item)
    if len(selected) < CASES_PER_PAIR:
        raise ValueError(f"Not enough mistakes for pair {pair}: {len(pool)}")
    return selected[:CASES_PER_PAIR]


def center_of_mass(image):
    ink = image.astype(np.float64)
    total = ink.sum()
    if not total:
        return float("nan"), float("nan")
    yy, xx = np.indices(ink.shape)
    return float((xx * ink).sum() / total), float((yy * ink).sum() / total)


def write_csv(path, rows):
    if not rows:
        raise ValueError(f"Cannot write empty file: {path}")
    with path.open("w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)


def main():
    predictions_path = BASELINE / "predictions.csv"
    metrics_path = BASELINE / "metrics.json"
    with predictions_path.open(encoding="utf-8-sig", newline="") as f:
        records = list(csv.DictReader(f))
    exp002 = json.loads(metrics_path.read_text(encoding="utf-8"))
    if len(records) != 2560 or not np.isclose(exp002["accuracy"], 0.58828125):
        raise ValueError("EXP-002 baseline metadata mismatch")
    for row in records:
        row["writer"] = int(row["writer"])
        row["index"] = int(row["index"])
        row["correct"] = int(row["correct"])
    images, labels = load_handmade()
    for row in records:
        true = LABELS[int(labels[row["writer"]][row["index"]])]
        if row["true"] != true or row["correct"] != int(row["true"] == row["predicted"]):
            raise ValueError(f"EXP-002 predictions and handwriting disagree: {row}")

    X_train, y_train, _, _, sha = fixed_split()
    if sha != exp002["official_dataset_sha256"] or len(y_train) != 76800:
        raise ValueError("Official train provenance differs from EXP-002")
    estimator, scaled = build_baseline()
    if not scaled:
        raise ValueError("Expected scaler-enabled KNN baseline")
    model = pipeline(estimator, scaled)
    print("Rebuilding unchanged KNN baseline (76,800 official training samples)...", flush=True)
    model.fit(X_train, y_train)
    scaler = model.named_steps["scale_0_1"]
    knn = model.named_steps["model"]

    OUTPUT.mkdir(parents=True, exist_ok=True)
    cases_rows, neighbors_rows = [], []
    for target_true, target_pred in TARGET_PAIRS:
        selected = select_cases(records, (target_true, target_pred))
        fig, axes = plt.subplots(CASES_PER_PAIR, 6, figsize=(13.5, 9))
        for case_num, r in enumerate(selected, start=1):
            writer, idx = r["writer"], r["index"]
            img = images[writer][idx]
            x_query = img.reshape(1, -1).astype(np.float32)
            predicted = LABELS[int(model.predict(x_query)[0])]
            if predicted != r["predicted"]:
                raise ValueError(f"EXP-002 prediction not reproduced: writer={writer}, index={idx}, {predicted} != {r['predicted']}")
            distances, indices = knn.kneighbors(scaler.transform(x_query), n_neighbors=5)
            distances, indices = distances[0], indices[0]
            labels_neighbors = [LABELS[int(y_train[i])] for i in indices]
            com_x, com_y = center_of_mass(img)
            case_id = f"{target_true}_to_{target_pred}_W{writer}_{idx:03d}"
            cases_rows.append({
                "case_id": case_id, "writer": writer, "index": idx,
                "true": target_true, "predicted": target_pred,
                "com_x": round(com_x, 3), "com_y": round(com_y, 3),
                "neighbor_labels": "/".join(labels_neighbors),
                "neighbor_votes": "/".join(f"{k}:{v}" for k, v in sorted(Counter(labels_neighbors).items())),
            })
            a = axes[case_num - 1, 0]
            a.imshow(img, cmap="gray", vmin=0, vmax=255)
            a.set_title(f"Hand-made W{writer} #{idx}\nTrue {target_true} / Pred {target_pred}", fontsize=9)
            a.axis("off")
            for rank, (dist, train_idx) in enumerate(zip(distances, indices), start=1):
                label = LABELS[int(y_train[train_idx])]
                a = axes[case_num - 1, rank]
                a.imshow(X_train[train_idx].reshape(28, 28), cmap="gray", vmin=0, vmax=255)
                a.set_title(f"Neighbor {rank}: {label}\nDistance {dist:.3f}", fontsize=9)
                a.axis("off")
                neighbors_rows.append({
                    "case_id": case_id, "rank": rank, "neighbor_train_index": int(train_idx),
                    "neighbor_label": label, "scaled_euclidean_distance": round(float(dist), 6),
                })
        fig.suptitle(f"EXP-003 KNN neighbors | true {target_true} -> predicted {target_pred}", fontsize=15)
        fig.tight_layout(rect=(0, 0, 1, .96))
        filename = f"neighbors_{target_true}_to_{target_pred}.png"
        fig.savefig(OUTPUT / filename, dpi=170)
        plt.close(fig)
        print(f"Saved {filename}: {len(selected)} hand-made mistakes x 5 nearest neighbors", flush=True)

    write_csv(OUTPUT / "selected_cases.csv", cases_rows)
    write_csv(OUTPUT / "nearest_neighbors.csv", neighbors_rows)
    summary = {
        "experiment": "EXP-003 KNN Normal misclassification + nearest-neighbor inspection",
        "input_baseline": "EXP002_handmade_baseline",
        "baseline_accuracy": exp002["accuracy"],
        "baseline_macro_f1": exp002["macro_f1"],
        "training_n": int(len(y_train)),
        "reviewed_pairs": [f"{a}->{b}" for a, b in TARGET_PAIRS],
        "cases_per_pair": CASES_PER_PAIR,
        "n_cases": len(cases_rows),
        "n_neighbors_per_case": 5,
        "note": "Selected cases illustrate errors. They are not a representative random sample. No causal claim or model improvement is established.",
    }
    (OUTPUT / "summary.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"Done: {len(cases_rows)} cases; {len(neighbors_rows)} nearest-neighbor comparisons")
    print("Saved:", OUTPUT)


if __name__ == "__main__":
    main()


## 실험 실행 및 결과 확인

`RUN_PLAN`에 실제 코드의 CLI 인자와 실행 순서를 넣고, `ALLOW_EXECUTION=True`를 설정한 경우에만 실행된다. 이전 결과를 다시 계산한 것과 새 학습 결과를 구별한다.

In [ ]:
if RUN_PLAN:
    for source_path, arguments in RUN_PLAN:
        execute_original(source_path, arguments)
else:
    print("원본 코드 등록 완료. 학습/평가는 실행하지 않았습니다.")